In [1]:
import sys
print(sys.executable)

C:\Users\chami\sepsis-early-warning\.venv\Scripts\python.exe


In [2]:
import sys
sys.path.append("..")
import duckdb
import pandas as pd
from src.columns import VITALS, LABS

con = duckdb.connect("../data/sepsis.duckdb", read_only=True)

def q(sql):
    """Run a SQL query and return a pandas DataFrame."""
    return con.sql(sql).df()

In [3]:
q("""
SELECT hospital,
       COUNT(DISTINCT patient_id) AS patients,
       COUNT(*)                   AS rows
FROM observations
GROUP BY hospital
ORDER BY hospital
""")

,hospital,patients,rows
0,A,20336,790215
1,B,20000,761995


In [4]:
q("""
SELECT COUNT(*) FILTER (WHERE isnan(HR)) AS hr_nan_values,
       COUNT(*) FILTER (WHERE HR IS NULL) AS hr_null_values
FROM observations
""")

,hr_nan_values,hr_null_values
0,0,153399


In [5]:
q("""
WITH p AS (
    SELECT patient_id, hospital,
           MAX(SepsisLabel) AS septic,
           COUNT(*)         AS n_rows,
           SUM(SepsisLabel) AS n_pos
    FROM observations
    GROUP BY ALL
)
SELECT hospital,
       COUNT(*)                            AS patients,
       SUM(septic)                         AS septic_patients,
       ROUND(AVG(septic), 4)               AS septic_patient_frac,
       SUM(n_rows)                         AS hours,
       ROUND(SUM(n_pos) / SUM(n_rows), 4)  AS positive_hour_frac
FROM p
GROUP BY hospital
ORDER BY hospital
""")

,hospital,patients,septic_patients,septic_patient_frac,hours,positive_hour_frac
0,A,20336,1790.0,0.0880,790215.0,0.0217
1,B,20000,1142.0,0.0571,761995.0,0.0141


In [6]:
q("""
WITH p AS (
    SELECT patient_id, hospital,
           MIN(ICULOS)            AS first_h,
           MAX(ICULOS)            AS last_h,
           COUNT(*)               AS n_rows,
           COUNT(DISTINCT ICULOS) AS n_distinct
    FROM observations
    GROUP BY ALL
)
SELECT hospital,
       SUM((first_h = 1)::INT)                    AS start_at_1,
       SUM((first_h > 1)::INT)                    AS start_later,
       MAX(first_h)                               AS latest_start,
       SUM((last_h - first_h + 1 <> n_rows)::INT) AS with_gaps,
       SUM((n_distinct <> n_rows)::INT)           AS with_duplicate_hours
FROM p
GROUP BY hospital
ORDER BY hospital
""")

,hospital,start_at_1,start_later,latest_start,with_gaps,with_duplicate_hours
0,A,12839.0,7497.0,304,0.0,0.0
1,B,18704.0,1296.0,26,0.0,0.0


In [7]:
q("""
SELECT COUNT(DISTINCT patient_id) AS patients_with_1_then_0
FROM (
    SELECT patient_id, SepsisLabel,
           LAG(SepsisLabel) OVER (PARTITION BY patient_id ORDER BY ICULOS) AS prev_label
    FROM observations
)
WHERE prev_label = 1 AND SepsisLabel = 0
""")

,patients_with_1_then_0
0,0


In [8]:
q("""
WITH s AS (
    SELECT patient_id, hospital,
           MIN(ICULOS) FILTER (WHERE SepsisLabel = 1) AS first_label_h,
           MAX(ICULOS)                                AS last_h
    FROM observations
    GROUP BY ALL
    HAVING MAX(SepsisLabel) = 1
)
SELECT hospital,
       last_h - first_label_h + 1 AS rows_from_first_label,
       COUNT(*)                   AS patients
FROM s
GROUP BY ALL
ORDER BY hospital, rows_from_first_label
""")

,hospital,rows_from_first_label,patients
0,A,6,2
1,A,7,2
2,A,8,93
3,A,9,564
4,A,10,1129
5,B,1,5
6,B,2,3
7,B,4,2
8,B,6,3
9,B,7,2


In [9]:
q("""
WITH p AS (
    SELECT patient_id, hospital,
           MAX(SepsisLabel) AS septic,
           COUNT(*)         AS n_rows
    FROM observations
    GROUP BY ALL
)
SELECT hospital, septic,
       COUNT(*)                        AS patients,
       MIN(n_rows)                     AS min_rows,
       quantile_cont(n_rows, 0.5)      AS median_rows,
       quantile_cont(n_rows, 0.95)     AS p95_rows,
       MAX(n_rows)                     AS max_rows,
       SUM((n_rows > 60)::INT)         AS longer_than_60
FROM p
GROUP BY ALL
ORDER BY hospital, septic
""")

,hospital,septic,patients,min_rows,median_rows,p95_rows,max_rows,longer_than_60
0,A,0,18546,8,39.0,56.00,336,6.0
1,A,1,1790,8,38.0,186.10,336,625.0
2,B,0,18858,8,38.0,57.00,336,218.0
3,B,1,1142,8,37.0,200.95,336,383.0


In [10]:
q("""
WITH s AS (
    SELECT patient_id, hospital,
           MIN(ICULOS)                                AS first_h,
           MIN(ICULOS) FILTER (WHERE SepsisLabel = 1) AS first_label_h
    FROM observations
    GROUP BY ALL
    HAVING MAX(SepsisLabel) = 1
)
SELECT hospital,
       COUNT(*)                                       AS septic_patients,
       SUM((first_label_h = first_h)::INT)            AS labeled_from_first_hour,
       SUM((first_label_h - first_h < 6)::INT)        AS under_6h_before_label,
       SUM((first_label_h - first_h < 12)::INT)       AS under_12h_before_label
FROM s
GROUP BY hospital
ORDER BY hospital
""")

,hospital,septic_patients,labeled_from_first_hour,under_6h_before_label,under_12h_before_label
0,A,1790,203.0,406.0,594.0
1,B,1142,223.0,327.0,399.0


In [13]:
q("""
WITH o AS (
    SELECT hospital, ICULOS,
           MAX(SepsisLabel) OVER (PARTITION BY patient_id)               AS septic,
           ICULOS - MIN(ICULOS) OVER (PARTITION BY patient_id) + 1       AS hours_observed
    FROM observations
)
SELECT hospital, septic,
       ROUND(AVG((ICULOS > 60)::INT), 3)         AS frac_rows_iculos_gt_60,
       ROUND(AVG((hours_observed > 60)::INT), 3) AS frac_rows_observed_gt_60
FROM o
GROUP BY ALL
ORDER BY hospital, septic
""")

,hospital,septic,frac_rows_iculos_gt_60,frac_rows_observed_gt_60
0,A,0,0.003,0.002
1,A,1,0.370,0.364
2,B,0,0.020,0.020
3,B,1,0.387,0.386


In [11]:
cols = VITALS + LABS
expr = ",\n".join(f"ROUND(AVG(({c} IS NULL)::INT), 3) AS {c}" for c in cols)

miss = q(f"SELECT hospital, {expr} FROM observations GROUP BY hospital ORDER BY hospital")
miss = miss.set_index("hospital").T          # one row per column, one column per hospital
miss["B_minus_A"] = (miss["B"] - miss["A"]).round(3)
miss.sort_values("A")

hospital,A,B,B_minus_A
HR,0.077,0.121,0.044
Resp,0.098,0.211,0.113
MAP,0.102,0.148,0.046
O2Sat,0.120,0.141,0.021
SBP,0.152,0.139,-0.013
DBP,0.481,0.139,-0.342
Temp,0.662,0.661,-0.001
FiO2,0.858,0.977,0.119
Glucose,0.878,0.778,-0.100
Hct,0.882,0.942,0.060


In [12]:
q("""
SELECT
    SUM((n_age    > 1)::INT) AS patients_age_changes,
    SUM((n_gender > 1)::INT) AS patients_gender_changes,
    SUM((n_unit1  > 1)::INT) AS patients_unit1_changes,
    SUM((n_unit2  > 1)::INT) AS patients_unit2_changes,
    SUM((n_hosp   > 1)::INT) AS patients_hospadm_changes
FROM (
    SELECT patient_id,
           COUNT(DISTINCT Age)         AS n_age,
           COUNT(DISTINCT Gender)      AS n_gender,
           COUNT(DISTINCT Unit1)       AS n_unit1,
           COUNT(DISTINCT Unit2)       AS n_unit2,
           COUNT(DISTINCT HospAdmTime) AS n_hosp
    FROM observations
    GROUP BY patient_id
)
""")

,patients_age_changes,patients_gender_changes,patients_unit1_changes,patients_unit2_changes,patients_hospadm_changes
0,0.0,0.0,0.0,0.0,0.0


In [14]:
q("""
SELECT hospital,
       ROUND(AVG(SepsisLabel), 4)                                AS overall_positive_rate,
       ROUND(AVG(SepsisLabel) FILTER (WHERE ICULOS > 60), 4)     AS positive_rate_after_60h,
       ROUND(AVG(SepsisLabel) FILTER (WHERE ICULOS <= 60), 4)    AS positive_rate_up_to_60h
FROM observations
GROUP BY hospital
ORDER BY hospital
""")

,hospital,overall_positive_rate,positive_rate_after_60h,positive_rate_up_to_60h
0,A,0.0217,0.1416,0.0152
1,B,0.0141,0.0858,0.0102


## Findings (Part E: full-data checks in DuckDB)

**Load and integrity**
- Patients: A 20,336, B 20,000 (total 40,336); rows: A 790,215, B 761,995 (total 1,552,210)
- NaN values after load: 0 (nullstr worked); HR missing in 153,399 rows (9.9%)
- No gaps and no duplicate hours in any record: within a record, rows are consecutive hours
- Static columns (Age, Gender, Unit1, Unit2, HospAdmTime) never change within a stay
- SepsisLabel never returns to 0 after turning 1 (0 exceptions)

**Class balance**
| | A | B |
|---|---|---|
| Septic patients | 1,790 (8.8%) | 1,142 (5.7%) |
| Positive hours | 2.2% | 1.4% |
- Sepsis prevalence differs between hospitals: a model trained on A will over-expect sepsis at B (calibration and PPV shift; Day 4)

**Record start (ICULOS)**
- Records starting after hour 1: A 7,497 (36.9%, latest start hour 304); B 1,296 (6.5%, latest 26)
- ICULOS ≠ hours observed, especially at A

**Truncation hypothesis: confirmed**
- Rows from first label to end of record: A: 10 rows 1,129 (63%), 9 rows 564 (32%), 8 rows 93 (5%), <8 rows 4. B: 10 rows 741 (65%), 9 rows 231 (20%), 8 rows 155 (14%), <8 rows 15
- ~99% of septic records end 1–3 h after onset. Total record length is leakage
- B has 10 records ending 1–4 rows after the first label (before the computed onset): odd cases to note

**Length-cap hypothesis: confirmed**
| | Median rows | 95th pct | Max | Records > 60 rows |
|---|---|---|---|---|
| A non-septic | 39 | 56 | 336 | 6 of 18,546 |
| A septic | 38 | 186 | 336 | 625 of 1,790 (35%) |
| B non-septic | 38 | 57 | 336 | 218 of 18,858 (1.2%) |
| B septic | 37 | 201 | 336 | 383 of 1,142 (34%) |
- All records capped at 336 h (14 days). Medians are equal; the difference is entirely in the long tail
- At A, a record longer than 60 rows is almost always septic: strong shortcut risk for any time-in-ICU feature

**Early-labeled septic patients (cohort decision)**
| | A | B |
|---|---|---|
| Labeled from first hour | 203 (11%) | 223 (20%) |
| < 6 h of data before label | 406 (23%) | 327 (29%) |
| < 12 h before label | 594 (33%) | 399 (35%) |

**Missingness: hospitals differ sharply**
- Blood gas labs nearly absent at B: pH 98%, PaCO2 98%, BaseExcess 99.8%, HCO3 99.8%, FiO2 98% missing (A: 86–91%)
- EtCO2 never measured at A (100% missing), 92% missing at B
- DBP missing 48% at A vs 14% at B; Resp 10% at A vs 21% at B; HR 8% vs 12%
- Glucose measured more often at B (78% vs 88% missing)
- Nearly empty everywhere (≥98% missing at both): Bilirubin_direct, Fibrinogen, TroponinI, Bilirubin_total
- Implication: features using these variables, especially missingness flags, will shift between hospitals; EtCO2 is unusable when training on A

**Shortcut check: ICULOS vs hours observed**
| | Rows with ICULOS > 60 | Rows with hours observed > 60 |
|---|---|---|
| A non-septic | 0.3% | 0.2% |
| A septic | 37.0% | 36.4% |
| B non-septic | 2.0% | 2.0% |
| B septic | 38.7% | 38.6% |
- Late starts are small offsets, so ICULOS ≈ hours observed; the shortcut exists in both
- Decision: exclude ICULOS from the main model; run a with/without comparison; use only fixed-window features (no cumulative-since-admission counts); do not restrict to hours ≤ 60 (would drop ~35% of septic patients' positive hours)
- Positive-label rate by time in ICU: A 1.5% (≤ 60 h) vs 14.2% (> 60 h), ~9×; B 1.0% vs 8.6%, ~8×. A model with ICULOS would learn a dataset artifact, not physiology. Confirms the decision to exclude it.